# 实战案例：基于注意力的图像描述

## 文件目录与提交规范

**本作业要求使用PyTorch实现。请补全标有 `# TODO` 的代码。**

所有路径均相对于本作业文件夹。数据需自行下载：https://pan.quark.cn/s/6ef11a2c284a

数据解压后固定放在 `data/flickr8k/`，其中包含 `dataset_flickr8k.json` 和 `images/`。生成的 `vocab.json`、`train_data.json`、`val_data.json` 也保存在 `data/flickr8k/`。

模型统一保存在 `work/` 文件夹中。运行时自动创建 `work/`。

### 提交清单

请提交以下文件，缺少任何一项将影响评分：

| 文件                         | 说明 |
|----------------------------|------|
| `caption课后作业.ipynb`        | 补全所有 TODO 并保留运行输出的 Notebook |
| `work/arctic_resnet101.pt` | 训练好的模型权重（含 `model`、`vocab`、`val_bleu4`） |

**不提交 `data/` 数据集。**

## 图像描述技术简介

图像描述的关键是生成自然语言描述图像中可以用语言表述的部分。传统的图像描述技术首先通过分析视觉内容来预测给定图像最可能包含的语义信息，并显式的转化为语言标签（通常为单词、短语或其他结构化描述），再基于这些标签生成自然语言描述句子。这类方法均使用以下的管道式结构实现图像描述任务： 

（1）使用计算机视觉技术来对场景进行分类，检测图像中存在的对象，预测它们的属性以及它们之间的关系，识别发生的动作，将它们映射为一些基本的自然语言描述单元，例如单词、短语或其他结构化描述。 

（2）通过自然语言生成技术（例如，模板，n-gram，语法规则等）将这些单词或者短语进行组合，生成自然语言描述句子。

这种管道式方法虽然充分利用了两个领域的现有技术，设计了一套简单可控的解决方案，然而，也存在若干问题：其一，分阶段的方式限制了两个模态数据间的信息交互；其二，这种方法高度依赖于预先定义的场景、对象、属性和动作的封闭语义类集；其三，这种分阶段的模型存在误差累积问题，前面任务的误差在后面阶段会放大；其四，训练误差不能前向传递。

当前主流的图像描述技术大多采用基于编解码框架的方法直接学习图像到文本描述的映射，其核心思想是建模一个以图像为条件的语言模型，计算视觉模式与文本模式的共现概率；其技术基础是深层神经网络对图文两种不同模态数据的通用表示学习能力，可以形成一个端到端的编码解码模型结构。此类方法中所使用的模型可以被端到端地训练，且不需要显示地定义图像和文本之间的桥梁（状态表示），可以有效避免前述管道式方法的问题。

不同的图像描述编解码模型的区别在于其图像编码器和文本解码器所使用的结构的不同。下表列举了深度学习时代常见的图像描述编解码器组合。

| 图像编码器 | 文本解码器 | 
| :----: | :----: | 
| 整体表示 | RNN | 
| 局部表示 | RNN+注意力 | 
| 局部表示+自注意力 | RNN+注意力 | 
| 局部表示+图网络 | RNN+注意力 | 
| 局部表示+Transformer编码器 | Transformer解码器 | 
| 视觉Transformer | Transformer解码模块 | 

接下来，我们将介绍一个图像编码器为CNN网格表示提取器、文本解码器为RNN+注意力的图像描述方法的具体实现。我们的实现大体上是在复现ARCTIC模型，但是在细节上有一些改变，下面的实现过程会对这些改变做具体说明。此外，[链接](https://github.com/sgrvinod/a-PyTorch-Tutorial-to-Image-Captioning)给出了一个更接近原始ARCTIC模型的代码库，非常推荐大家阅读。本节的部分代码也是受到该代码库的启发。

下面，按照读取数据、定义模型、定义损失函数、选择优化方法、选择评估指标和训练模型的次序，来描述该实战案例。

c## 1. 环境初始化与数据定位

本单元导入 PyTorch、NumPy、PIL 等依赖，固定随机种子并定位作业目录。默认以当前工作目录作为根目录，也可通过代码中的作业目录环境变量指定根目录。

请提前将图片和划分文件整理到 `data/flickr8k/`：`images/` 存放图片，`dataset_flickr8k.json` 保存图片、caption 和官方 split。程序会检查这两个输入；缺失时会提示错误。本单元不执行解压操作。

使用完整的 Flickr8k 数据集（Karpathy split），每图5条描述，原始文本最多保留30个 token。运行后应先核对打印的数据来源。

In [ ]:
from pathlib import Path
import json
import math
import os
import random
from collections import Counter, defaultdict
from types import SimpleNamespace

import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models
import torchvision.transforms as tv_transforms

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

device = torch.device("cuda")
configured_root = os.environ.get("HOMEWORK2_DIR")
NOTEBOOK_DIR = (
    Path(configured_root).expanduser().resolve()
    if configured_root else Path.cwd().resolve()
)
SOURCE_DIR = NOTEBOOK_DIR / "data" / "flickr8k"
if not (SOURCE_DIR / "dataset_flickr8k.json").is_file() or not (SOURCE_DIR / "images").is_dir():
    raise FileNotFoundError(
        f"请将数据放在 {SOURCE_DIR}，该目录须包含 dataset_flickr8k.json 和 images/。"
    )
OUTPUT_DIR = SOURCE_DIR
CAPTIONS_PER_IMAGE = 5
MAX_LEN = 30
print(f"使用设备: {device}")
print(f"数据来源: {SOURCE_DIR}")

## 2. 整理数据集与构建词表

原始 JSON 中的 `split` 定义了训练、验证和测试划分（Karpathy split）。使用每个 split 的全部图片，将自然语言描述编码为整数序列；图片只记录路径，读取样本时再加载图像。

词表统计训练集和验证集的 caption（测试集不参与），保留出现至少5次的词，并加入 `<pad>`、`<unk>`、`<start>`、`<end>`。测试中的未登录词映射为 `<unk>`。文本截断后在首尾加入开始和结束标记。

输出 `vocab.json`、`train_data.json`、`val_data.json`，均保存在 `data/flickr8k/`。

**任务与提示：** 对照 TODO 1 完成数据集整理，理解图片列表与 caption 列表的对应关系，并核对生成数量。

In [ ]:
def create_dataset(source_dir, output_dir, captions_per_image=5, min_word_count=5, max_len=30):
    """按 Karpathy split 使用全部图片，构建词表和数据集文件。"""
    source_dir = Path(source_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    raw = json.loads((source_dir / "dataset_flickr8k.json").read_text())

    image_paths = defaultdict(list)
    image_captions = defaultdict(list)
    vocab_counter = Counter()

    for item in raw["images"]:
        split = item.get("split")
        if split not in ("train", "val"):
            continue
        captions = []
        for sent in item["sentences"]:
            vocab_counter.update(sent["tokens"])
            if len(sent["tokens"]) <= max_len:
                captions.append(sent["tokens"])
        if len(captions) == 0:
            continue
        image_path = source_dir / "images" / item["filename"]
        if not image_path.is_file():
            raise FileNotFoundError(image_path)
        image_paths[split].append(str(image_path))
        image_captions[split].append(captions)

    # STUDENT_TODO 1：构建词表，保留出现次数超过 min_word_count 的词
    words = sorted(w for w, c in vocab_counter.items() if c > min_word_count)
    vocab = {"<pad>": 0, "<unk>": 1, "<start>": 2, "<end>": 3}
    vocab.update({word: idx + 4 for idx, word in enumerate(words)})
    (output_dir / "vocab.json").write_text(json.dumps(vocab, ensure_ascii=False))

    summary = {}
    for split in ("train", "val"):
        paths = image_paths[split]
        caps_list = image_captions[split]
        encoded_captions = []
        for i, path in enumerate(paths):
            caps = caps_list[i]
            if len(caps) < captions_per_image:
                caps = caps + [random.choice(caps) for _ in range(captions_per_image - len(caps))]
            else:
                caps = random.sample(caps, k=captions_per_image)
            for cap in caps:
                encoded = [vocab["<start>"]]
                encoded += [vocab.get(w, vocab["<unk>"]) for w in cap]
                encoded += [vocab["<end>"]]
                encoded_captions.append(encoded)
        assert len(paths) * captions_per_image == len(encoded_captions)
        payload = {"IMAGES": paths, "CAPTIONS": encoded_captions}
        (output_dir / f"{split}_data.json").write_text(json.dumps(payload))
        summary[split] = (len(paths), len(encoded_captions))
    return vocab, summary


if not (OUTPUT_DIR / "vocab.json").is_file():
    vocab, split_summary = create_dataset(SOURCE_DIR, OUTPUT_DIR, CAPTIONS_PER_IMAGE, 5, MAX_LEN)
    print("数据集创建完成:", split_summary, "vocab:", len(vocab))
else:
    vocab = json.loads((OUTPUT_DIR / "vocab.json").read_text())
    split_summary = {}
    for split in ("train", "val"):
        data = json.loads((OUTPUT_DIR / f"{split}_data.json").read_text())
        split_summary[split] = (len(data["IMAGES"]), len(data["CAPTIONS"]))
    print("数据集已存在:", split_summary, "vocab:", len(vocab))

## 3. 定义 Dataset 与批量读取

自定义数据集继承 `torch.utils.data.Dataset`，实现 `__len__` 和 `__getitem__`。`ImageTextDataset` 以 caption 为样本单位；每张图片连续对应5条 caption，因此 caption 下标整除5得到图片下标。

图像转换为 RGB，缩放到256后裁切到 224×224，使用 ImageNet 归一化。caption 用 `<pad>` 补齐至固定长度，同时返回包含开始与结束标记的真实长度。

`make_loaders` 为训练、验证和测试分别构造 DataLoader；训练打乱顺序，评估保持顺序。

**任务与提示：** TODO 2 需要完成 caption 到图片的索引映射；不要把 caption 数误认为唯一图片数。

In [ ]:
IMG_SIZE = 224

train_transform = tv_transforms.Compose([
    tv_transforms.Resize(256),
    tv_transforms.RandomCrop(IMG_SIZE),
    tv_transforms.ToTensor(),
    tv_transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = tv_transforms.Compose([
    tv_transforms.Resize(256),
    tv_transforms.CenterCrop(IMG_SIZE),
    tv_transforms.ToTensor(),
    tv_transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def image_to_tensor(path, transform=None):
    image = Image.open(path).convert("RGB")
    if transform is not None:
        return transform(image)
    return val_transform(image)


class ImageTextDataset(Dataset):
    def __init__(self, data_path, vocab_path, captions_per_image=5, max_len=30, transform=None):
        self.data = json.loads(Path(data_path).read_text())
        self.vocab = json.loads(Path(vocab_path).read_text())
        self.cpi = captions_per_image
        self.max_len = max_len
        self.transform = transform

    def __len__(self):
        return len(self.data["CAPTIONS"])

    def __getitem__(self, index):
        # STUDENT_TODO 2：由 caption 下标映射到 image 下标
        image_index = index // self.cpi
        image = image_to_tensor(self.data["IMAGES"][image_index], self.transform)
        tokens = self.data["CAPTIONS"][index]
        length = len(tokens)
        padded = tokens + [self.vocab["<pad>"]] * (self.max_len + 2 - length)
        return image, torch.tensor(padded, dtype=torch.long), length


def make_loaders(output_dir, batch_size=32):
    output_dir = Path(output_dir)
    vocab_path = output_dir / "vocab.json"
    generator = torch.Generator().manual_seed(SEED)
    train_ds = ImageTextDataset(
        output_dir / "train_data.json", vocab_path,
        CAPTIONS_PER_IMAGE, MAX_LEN, transform=train_transform,
    )
    val_ds = ImageTextDataset(
        output_dir / "val_data.json", vocab_path,
        CAPTIONS_PER_IMAGE, MAX_LEN, transform=val_transform,
    )
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                   num_workers=0, generator=generator),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0),
    )

## 4. 图像编码器

使用 torchvision 提供的在 ImageNet 上预训练的 ResNet-101 作为图像编码器。去掉最后的平均池化层和全连接层，保留卷积特征图。输出形状为 `[B, 2048, 7, 7]`，即 49 个空间区域，每个区域 2048 维特征。

`grid=True` 返回局部网格特征（供注意力模型使用）；`grid=False` 返回全局向量。图像描述使用 `grid=True`。

**任务与提示：** TODO 3 完成 ResNet-101 特征提取器的初始化与前向传播。需要去掉最后两层（AvgPool 和 FC），保留卷积特征图。

In [ ]:
class ImageEncoder(nn.Module):
    """预训练 ResNet-101 图像编码器。"""
    def __init__(self, grid=True, finetuned=True):
        super().__init__()
        self.grid = grid
        resnet = tv_models.resnet101(weights=tv_models.ResNet101_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*(list(resnet.children())[:-2]))
        for param in self.features.parameters():
            param.requires_grad = finetuned
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, images):
        # STUDENT_TODO 3：提取卷积特征并按 grid 参数返回网格或全局特征
        features = self.features(images)
        if self.grid:
            return features
        else:
            return self.global_pool(features).flatten(1)

## 5. Additive Attention、文本解码器与 ARCTIC

ARCTIC 风格模型将图像局部特征与循环文本解码器结合。在每个生成时刻，模型根据当前 hidden state 选择相关视觉区域，再预测下一个词。

### 加性注意力

查询为 hidden state，键和值为49个图像区域向量（7×7 网格）：

$$e_j=w^T\tanh(W_q q+W_k k_j),\quad
\alpha_j=\mathrm{softmax}_j(e_j),\quad c=\sum_j\alpha_j k_j.$$

分别映射 query 和 keys，相加后经过 tanh 和评分层，沿区域维度归一化，再加权求和得到 context。

**任务与提示：**
- **TODO 4：** 在 `AdditiveAttention.forward` 中实现加性注意力的 energy 计算。将 query 和 key_value 分别映射后相加，经 tanh 和 score 层得到标量。
- **TODO 5：** 在 `AttentionDecoder.forward` 中实现动态批大小的解码循环。每个时间步只处理 `lengths_np > step` 的样本，避免 padding 参与运算。
- **TODO 6：** 在 `ARCTIC.generate_by_beamsearch` 中实现束搜索生成。对每张图片维护 k 个候选序列，每步扩展并保留全局 top-k，遇到 `<end>` 存入完成列表，最终选概率最高的完整序列。

### 动态批大小的训练解码流程

1. 将网格展开为 `[B, 49, 2048]`，**按 caption 长度从长到短排序**。
2. 由区域均值初始化 hidden state。
3. 取人工 caption 的当前词 embedding，采用 teacher forcing。
4. 计算 attention context，与词向量拼接后输入 GRUCell。
5. **每个时间步只处理当前仍有有效 token 的样本**（动态缩减 batch），避免 `<pad>` 参与运算。
6. 分类层输出词表上的 logits，逐步收集预测值与 attention 权重。

### 生成与模型组装

`ARCTIC.forward` 连接 ImageEncoder 与 AttentionDecoder。生成接口包含贪心解码和**束搜索解码**（beam search，k=5）。束搜索在每一步保留概率最高的 k 个候选序列，遇到 `<end>` 的候选存入完成列表，最终选取概率最高的完整序列。

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, query_dim, key_dim, attention_dim):
        super().__init__()
        self.query_proj = nn.Linear(query_dim, attention_dim)
        self.key_proj = nn.Linear(key_dim, attention_dim)
        self.score = nn.Linear(attention_dim, 1)

    def forward(self, query, key_value):
        # STUDENT_TODO 4：实现 Additive Attention 的 energy
        energy = self.score(torch.tanh(
            self.query_proj(query).unsqueeze(1) + self.key_proj(key_value)
        )).squeeze(-1)
        alpha = torch.softmax(energy, dim=1)
        context = (alpha.unsqueeze(-1) * key_value).sum(dim=1)
        return context, alpha


class AttentionDecoder(nn.Module):
    def __init__(self, image_dim, vocab_size, word_dim=512, hidden_dim=512, attention_dim=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, word_dim, padding_idx=0)
        self.attention = AdditiveAttention(hidden_dim, image_dim, attention_dim)
        self.init_hidden = nn.Linear(image_dim, hidden_dim)
        self.gru = nn.GRUCell(word_dim + image_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, vocab_size)
        self.vocab_size = vocab_size

    def prepare_image(self, image_grid):
        return image_grid.flatten(2).transpose(1, 2)

    def init_hidden_state(self, image_code, captions, lengths):
        """按 caption 长度从长到短排序，初始化隐状态。"""
        regions = self.prepare_image(image_code)
        sorted_lengths, sorted_indices = lengths.sort(descending=True)
        regions = regions[sorted_indices]
        captions = captions[sorted_indices]
        hidden = torch.tanh(self.init_hidden(regions.mean(dim=1)))
        return regions, captions, sorted_lengths, sorted_indices, hidden

    def forward_step(self, regions, word_embed, hidden):
        """单步解码：注意力 -> GRU -> 预测。"""
        context, alpha = self.attention(hidden, regions)
        hidden = self.gru(torch.cat([word_embed, context], dim=1), hidden)
        preds = self.classifier(hidden)
        return preds, alpha, hidden

    def forward(self, image_grid, captions, lengths):
        regions, captions, sorted_lengths, sorted_indices, hidden = \
            self.init_hidden_state(image_grid, captions, lengths)
        embeddings = self.embedding(captions)
        max_steps = int(sorted_lengths[0].item()) - 1
        batch_size = regions.size(0)
        predictions = torch.zeros(batch_size, max_steps, self.vocab_size, device=regions.device)
        alphas = torch.zeros(batch_size, max_steps, regions.size(1), device=regions.device)
        lengths_np = sorted_lengths.cpu().numpy() - 1
        # STUDENT_TODO 5：实现动态批大小的解码循环
        for step in range(max_steps):
            real_batch = int(np.where(lengths_np > step)[0].shape[0])
            preds, alpha, hidden_out = self.forward_step(
                regions[:real_batch], embeddings[:real_batch, step], hidden[:real_batch])
            hidden = hidden.clone()
            hidden[:real_batch] = hidden_out
            predictions[:real_batch, step] = preds
            alphas[:real_batch, step] = alpha
        return predictions, alphas, captions, lengths_np, sorted_indices

    def greedy_decode(self, image_grid, start_id, end_id, max_len=30):
        regions = self.prepare_image(image_grid)
        hidden = torch.tanh(self.init_hidden(regions.mean(dim=1)))
        current = torch.full((regions.size(0),), start_id, dtype=torch.long, device=regions.device)
        sequences = [[] for _ in range(regions.size(0))]
        finished = torch.zeros(regions.size(0), dtype=torch.bool, device=regions.device)
        for _ in range(max_len):
            context, _ = self.attention(hidden, regions)
            hidden = self.gru(torch.cat([self.embedding(current), context], dim=1), hidden)
            current = self.classifier(hidden).argmax(dim=1)
            for i, token in enumerate(current.tolist()):
                if not finished[i]:
                    sequences[i].append(token)
            finished |= current.eq(end_id)
            if finished.all():
                break
        return sequences


class ARCTIC(nn.Module):
    def __init__(self, image_code_dim, vocab, word_dim=512, attention_dim=512,
                 hidden_dim=512):
        super().__init__()
        self.vocab = vocab
        self.encoder = ImageEncoder(grid=True)
        self.decoder = AttentionDecoder(image_code_dim, len(vocab), word_dim, hidden_dim, attention_dim)

    def forward(self, images, captions, lengths):
        return self.decoder(self.encoder(images), captions, lengths)

    # STUDENT_TODO 6：实现束搜索生成
    def generate_by_beamsearch(self, images, beam_k, max_len):
        vocab_size = len(self.vocab)
        image_codes = self.encoder(images)
        texts = []
        for image_code in image_codes:
            image_code = image_code.unsqueeze(0).expand(beam_k, -1, -1, -1)
            regions = self.decoder.prepare_image(image_code)
            hidden = torch.tanh(self.decoder.init_hidden(regions.mean(dim=1)))

            cur_sents = torch.full((beam_k, 1), self.vocab["<start>"],
                                   dtype=torch.long, device=regions.device)
            cur_embed = self.decoder.embedding(cur_sents)[:, 0, :]

            end_sents, end_probs = [], []
            probs = torch.zeros(beam_k, 1, device=regions.device)
            k = beam_k

            while True:
                preds, _, hidden_new = self.decoder.forward_step(
                    regions[:k], cur_embed, hidden[:k])
                hidden = hidden.clone()
                hidden[:k] = hidden_new
                log_preds = F.log_softmax(preds, dim=1)
                probs_expanded = probs[:k].expand_as(log_preds) + log_preds

                if cur_sents.size(1) == 1:
                    values, indices = probs_expanded[0].topk(k, dim=0, largest=True, sorted=True)
                else:
                    values, indices = probs_expanded.reshape(-1).topk(k, dim=0, largest=True, sorted=True)

                sent_indices = indices // vocab_size
                word_indices = indices % vocab_size

                cur_sents = torch.cat([cur_sents[sent_indices], word_indices.unsqueeze(1)], dim=1)

                end_indices = [idx for idx, w in enumerate(word_indices.tolist())
                               if w == self.vocab["<end>"]]
                if end_indices:
                    end_probs.extend(values[end_indices].tolist())
                    end_sents.extend(cur_sents[end_indices].tolist())
                    k -= len(end_indices)
                    if k == 0:
                        break

                cur_indices = [idx for idx, w in enumerate(word_indices.tolist())
                               if w != self.vocab["<end>"]]
                if cur_indices:
                    cur_indices_t = torch.tensor(cur_indices, device=regions.device)
                    cur_sents = cur_sents[cur_indices_t]
                    probs = values[cur_indices_t].unsqueeze(1)
                    hidden_selected = hidden[sent_indices[cur_indices_t]]
                    hidden = hidden.clone()
                    hidden[:k] = hidden_selected
                    regions_selected = regions[sent_indices[cur_indices_t]]
                    regions = regions.clone()
                    regions[:k] = regions_selected
                    cur_embed = self.decoder.embedding(word_indices[cur_indices_t].unsqueeze(1))[:, 0, :]

                if cur_sents.size(1) >= max_len:
                    break

            if not end_sents:
                gen_sent = cur_sents[0].tolist()
            else:
                gen_sent = end_sents[end_probs.index(max(end_probs))]
            texts.append(gen_sent)
        return texts

## 6. 损失函数、训练参数与评估

### 损失与优化

由于解码器按长度排序并使用动态批大小，损失函数需要将各样本的有效预测拼接后统一计算交叉熵（TODO 7）。`attention_coverage_loss` 同样排除 padding 时间步，再计算每个区域累计 attention 与1的平方差。

总损失为交叉熵加 `1.0 × attention_coverage_loss`。使用 Adam，梯度范数裁剪阈值为5。

### 参数设置

```python
config = SimpleNamespace(
    batch_size=32, epochs=10, learning_rate=5e-4,
    image_code_dim=2048, word_dim=512, hidden_dim=512,
    attention_dim=512, beam_k=5
)
```

即 batch size 为 32，共训练 **10 轮**，学习率为 **0.0005**，使用 ResNet-101 编码器（2048 维），束搜索宽度 k=5。

### 流程与 BLEU-4

读取 batch → 前向传播→ 损失与反向传播 → 梯度裁剪 → Adam 更新。每轮结束使用**束搜索**在验证集上生成描述并计算 BLEU-4，保存验证指标最好的参数副本。

评估对每张图片通过束搜索生成一条描述，对应5条人工 reference。去掉开始、结束和 padding 标记后，`corpus_bleu4` 汇总全语料1至4阶 clipped n-gram counts，结合长度惩罚计算分数。

**注意：** 测试集不包含参考描述（captions），学生只能在验证集上计算 BLEU-4。**测试集 BLEU-4 由教师在批改时使用完整参考描述评估。**

**任务与提示：**
- **TODO 7：** 在 `caption_loss` 中实现损失计算。遍历每个样本，取 `predictions[i, :lengths[i]]` 和 `targets[i, :lengths[i]]`，拼接所有有效时间步后调用 `F.cross_entropy`。
- **TODO 8：** 在代码底部设置训练超参数 `config`，包括 `batch_size`、`epochs`、`learning_rate`、`image_code_dim`、`word_dim`、`hidden_dim`、`attention_dim`、`beam_k`。需要自行选择合理的值。

In [ ]:
def caption_loss(predictions, targets, lengths):
    """将各样本有效时间步的预测拼接后计算交叉熵。"""
    # STUDENT_TODO 7：拼接有效预测和目标，计算交叉熵
    preds_list, gts_list = [], []
    for i in range(len(lengths)):
        preds_list.append(predictions[i, :lengths[i], :])
        gts_list.append(targets[i, :lengths[i]])
    return F.cross_entropy(torch.cat(preds_list, dim=0), torch.cat(gts_list, dim=0))


def attention_coverage_loss(alphas, lengths):
    max_steps = alphas.size(1)
    valid_steps = (
        torch.arange(max_steps, device=alphas.device).unsqueeze(0)
        < torch.tensor(lengths, device=alphas.device).unsqueeze(1)
    )
    coverage = (alphas * valid_steps.unsqueeze(-1).float()).sum(dim=1)
    return ((1.0 - coverage) ** 2).mean()


def train_caption_epoch(loader, model, optimizer):
    model.train()
    total = 0.0
    for images, captions, lengths in loader:
        images, captions, lengths = images.to(device), captions.to(device), lengths.to(device)
        optimizer.zero_grad()
        predictions, alphas, sorted_captions, sorted_lengths, sorted_indices = \
            model(images, captions, lengths)
        loss = caption_loss(predictions, sorted_captions[:, 1:], sorted_lengths)
        loss = loss + 1.0 * attention_coverage_loss(alphas, sorted_lengths)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total += loss.item() * images.size(0)
    return total / len(loader.dataset)


def corpus_bleu4(all_references, hypotheses):
    """标准 corpus BLEU-4：在整个语料上聚合 clipped n-gram counts。"""
    clipped = [0, 0, 0, 0]
    totals = [0, 0, 0, 0]
    hypothesis_length = 0
    reference_length = 0
    for references, hypothesis in zip(all_references, hypotheses):
        hypothesis_length += len(hypothesis)
        reference_length += min(
            (len(ref) for ref in references),
            key=lambda length: (abs(length - len(hypothesis)), length),
        )
        for n in range(1, 5):
            hyp_counts = Counter(
                tuple(hypothesis[i:i+n])
                for i in range(max(0, len(hypothesis) - n + 1))
            )
            max_ref_counts = Counter()
            for reference in references:
                ref_counts = Counter(
                    tuple(reference[i:i+n])
                    for i in range(max(0, len(reference) - n + 1))
                )
                for gram, count in ref_counts.items():
                    max_ref_counts[gram] = max(max_ref_counts[gram], count)
            clipped[n-1] += sum(
                min(count, max_ref_counts[gram]) for gram, count in hyp_counts.items()
            )
            totals[n-1] += sum(hyp_counts.values())
    if hypothesis_length == 0 or any(value == 0 for value in clipped):
        return 0.0
    precisions = [match / total for match, total in zip(clipped, totals)]
    brevity = min(1.0, math.exp(1.0 - reference_length / hypothesis_length))
    return brevity * math.exp(sum(math.log(p) for p in precisions) / 4.0)


def evaluate_caption(data_path, model, beam_k=5, batch_size=16):
    payload = json.loads(Path(data_path).read_text())
    images = torch.stack([image_to_tensor(path) for path in payload["IMAGES"]])
    hypotheses = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            batch_imgs = images[start:start+batch_size].to(device)
                texts = model.generate_by_beamsearch(batch_imgs, beam_k, MAX_LEN + 2)
            hypotheses.extend(texts)
    all_references = []
    cleaned_hypotheses = []
    special = set(v for k, v in model.vocab.items() if k in ("<pad>", "<start>", "<end>"))
    for i, hypothesis in enumerate(hypotheses):
        begin = i * CAPTIONS_PER_IMAGE
        refs = payload["CAPTIONS"][begin:begin + CAPTIONS_PER_IMAGE]
        all_references.append([[token for token in ref if token not in special] for ref in refs])
        cleaned_hypotheses.append([token for token in hypothesis if token not in special])
    return corpus_bleu4(all_references, cleaned_hypotheses)


# STUDENT_TODO 8：设置训练超参数
config = SimpleNamespace(
    batch_size=32, epochs=10, learning_rate=5e-4,
    image_code_dim=2048, word_dim=512, hidden_dim=512,
    attention_dim=512, beam_k=5,
)
train_loader, val_loader = make_loaders(OUTPUT_DIR, config.batch_size)
model = ARCTIC(config.image_code_dim, vocab, config.word_dim,
               config.attention_dim, config.hidden_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
best_state, best_bleu = None, -1.0
for epoch in range(1, config.epochs + 1):
    loss = train_caption_epoch(train_loader, model, optimizer)
    val_bleu = evaluate_caption(OUTPUT_DIR / "val_data.json", model, config.beam_k)
    print(f"epoch {epoch}/{config.epochs}: loss={loss:.4f}, val_BLEU-4={val_bleu:.4f}")
    if val_bleu >= best_bleu:
        best_bleu = val_bleu
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)

checkpoint_dir = NOTEBOOK_DIR / "work"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
torch.save({"model": best_state, "vocab": vocab, "val_bleu4": best_bleu},
           checkpoint_dir / "arctic_resnet101.pt")
print(f"Best val BLEU-4={best_bleu:.4f}")
print("测试集 BLEU-4 由教师批改时评估。")

## 理解与实验分析（20分）

请结合实际代码和运行结果回答。

### 1. 词表构建策略（5分）

解释词频过滤阈值 `min_word_count=5` 的作用：过滤低频词对词表大小和模型训练有什么影响？四个特殊标记 `<pad>`、`<unk>`、`<start>`、`<end>` 分别在模型中承担什么角色？如果将阈值从5改为1（即保留所有词），可能会带来哪些问题？

**回答：**

词频过滤阈值 `min_word_count=5` 的作用是只保留在训练集和验证集中出现超过5次的词。这样做的好处：
1. **减小词表规模**：过滤大量低频词（如拼写错误、专有名词），使分类层（`nn.Linear(hidden_dim, vocab_size)`）更紧凑，减少参数量和过拟合风险。
2. **提升训练质量**：低频词的 embedding 难以充分训练（样本太少），保留它们会引入噪声。

四个特殊标记的角色：
- `<pad>`（索引0）：用于将不等长的 caption 补齐到固定长度，`padding_idx=0` 保证其 embedding 始终为零向量，不参与梯度更新。
- `<unk>`（索引1）：替代词表外的未登录词，使模型能处理未见过的词汇。
- `<start>`（索引2）：解码器的起始信号，告诉模型开始生成序列。
- `<end>`（索引3）：序列结束标记，束搜索遇到它时将候选存入完成列表。

如果将阈值改为1（保留所有词）：
1. 词表膨胀，分类层参数增多，训练更慢且更容易过拟合。
2. 大量词只出现1-2次，其 embedding 无法有效学习，生成时可能产生不相关的词。
3. Flickr8k 数据集较小，保留所有词会使词表中相当比例的词缺乏足够训练样本。

### 2. 注意力与变长序列（5分）

解释49个区域的 attention 归一化维度，说明动态批大小如何避免 padding 参与运算。

<cell_type>markdown</cell_type>**回答：**

ResNet-101 输出 `[B, 2048, 7, 7]` 的特征图，展平后得到 49 个区域向量 `[B, 49, 2048]`。Attention 的 softmax 沿区域维度（dim=1，即 49 个空间位置）归一化，使得每个时间步的注意力权重在所有区域上的和为 1，表示当前生成词对各区域的关注程度。

交叉熵和 coverage loss 都需要长度 mask 的原因：同一 batch 中不同 caption 的长度不同，短 caption 在后面的时间步是 `<pad>` 填充。如果不用 mask：
1. 交叉熵会把 padding 位置的预测也算入损失，模型被迫学习预测无意义的 `<pad>`，干扰真正的词汇学习。
2. Coverage loss 要求每个区域在所有有效时间步的注意力权重之和接近 1。如果把 padding 时间步也算入累加，短序列的 coverage 会偏小，产生不公平的惩罚。

本实现通过动态批大小进一步优化：按长度排序后，每个时间步只处理仍有有效 token 的样本，短序列在其结束后不再参与计算，既节省计算资源，又天然避免了 padding 干扰。

### 3. 训练配置与生成（5分）

指出 epochs 与 learning_rate 的设置位置，解释 teacher forcing 与 beam search 的区别。

<cell_type>markdown</cell_type>**回答：**

`epochs` 和 `learning_rate` 在第 6 节代码单元底部的 `config = SimpleNamespace(batch_size=32, epochs=10, learning_rate=5e-4, ...)` 中设置。Adam 优化器读取 `config.learning_rate`，训练循环读取 `config.epochs`。

Teacher forcing 与 beam search decode 的区别：
- **Teacher forcing**（训练时）：解码器每一步的输入是人工 caption 中的上一个真实词（`embeddings[:, step]`），而非模型自身的预测。这保证训练信号稳定，加速收敛。
- **Beam search**（推理时）：从 `<start>` 开始，每一步保留概率最高的 k 个候选序列，对每个候选扩展所有可能的下一个词并保留全局 top-k，遇到 `<end>` 的候选存入完成列表，最终选取概率最高的完整序列。相比 greedy decode 只选最大概率的单一路径，beam search 在更大搜索空间中寻找全局最优，生成质量更高。

### 4. 结果与失败案例（5分）

填写两轮 loss 和验证 BLEU-4。可新增展示单元给出一张验证集图片、生成描述与人工参考描述对比，分析对象、属性或关系错误。

**回答：**

（以下数值需以实际运行结果为准，此处为预期说明）

使用预训练 ResNet-101 编码器（2048 维，49 个空间区域），配合动态批大小和束搜索解码（k=5），在完整 Flickr8k 数据集上训练 10 个 epoch。

训练过程中 loss 持续下降，验证集 BLEU-4 逐步提高。束搜索相比贪心解码可以带来明显的 BLEU-4 提升，因为它能在更大的搜索空间中找到全局更优的描述序列。

典型的失败案例包括：
1. **对象遗漏**：图像中有多个物体时，生成描述可能只提到其中一个。
2. **属性错误**：颜色、大小等属性描述不准确（如"white dog"说成"black dog"）。
3. **关系错误**：物体间的空间或动作关系描述错误（如"standing"说成"sitting"）。

这些问题的原因：(1) Flickr8k 数据集规模较小（6000张训练图），模型对稀有组合的泛化能力有限；(2) CNN 网格特征以均匀划分的方式表示图像，可能无法精确捕捉小物体或细粒度属性；(3) GRU 解码器的序列建模能力弱于 Transformer，长距离依赖处理不够好。